# Litmus Test: BackwardStale Predicate

**Formal definition:** `BackwardStale(W, W', i, j) ≝ j < i ∧ j = LastWriter(W, i, y) for some y ∈ Wᵢ \ W'ᵢ`

Cell j (before i) becomes stale if j was the last writer of a location that cell i no longer writes. This handles the case where re-running a cell changes what it outputs.

**Test scenario:**
1. Run A (writes x=1, writes y=2)
2. Run B (reads y)
3. Modify A to only write x (remove y=2)
4. Re-run A

**Expected:** A no longer writes y, but B read from A's y. Since A was the LastWriter of y for B, and A no longer writes y, this triggers BackwardStale. However, since B is AFTER A, this actually triggers ForwardStale instead.

**Corrected scenario for BackwardStale:**
1. Run B (writes y=10)
2. Run A (writes y=20, overwriting B)
3. Modify A to no longer write y
4. Re-run A

**Expected:** B becomes stale because B was the LastWriter of y before A overwrote it, and now A no longer writes y (so B's write becomes relevant again but stale).

In [ ]:
# Cell B: Writes y (run FIRST)
y = 10
print(f"B: y = {y}")

In [ ]:
# Cell A: Writes y (run SECOND, overwrites B's y)
# After running B→A, modify this to just: x = 5
y = 20
print(f"A: y = {y}")

In [ ]:
# Cell C: Reads y (run THIRD)
# If A no longer writes y, C's source reverts to B
z = y * 2
print(f"z = {z}")